<a id="top"></a>
# Synced selection and filtering between `mast-table` and `mast-aladin`

This tutorial demonstrates how selections and filters set in `mast-table` can be synced to show in `mast-aladin`, and how selections in `mast-aladin` are then synced to show in the `mast-aladin`.  


This tutorial demonstrates how to synchronize selections between `mast-table` and `mast-aladin`, as well as how filters applied in `mast-table` can be reflected in `mast-aladin`


This tutorial demonstrates how to sync `mast-aladin` and `mast-table` so that:
- selections and filters applied in `mast-table` are reflected in `mast-aladin`
- selections applied in `mast-aladin` are reflected in `mast-table


## Imports

In [ ]:
from astropy.coordinates import SkyCoord
from astropy.table import Table

from astroquery.mast import MastMissions, conf

from mast_aladin import MastAladin
from mast_table import MastTable

<div class="alert alert-warning">
Warning: the following cells rely on Roman test data. If a user lacks the proper data permissions, these cells will NOT run.
</div>

## Search for Roman test data

In [ ]:
# load a previously-saved dataset from the Roman test environment, or
# query for the necessary data if dataset does not exist

target = SkyCoord(ra=270, dec=66, unit='deg')
    
try:
    roman_obs = Table.read("roman_obs.csv")
except:
    # configure MastMissions to be able to query Roman test data
    conf.server = "https://masttest.stsci.edu"
    mast = MastMissions(mission='roman')
    mast._service_api_connection.SERVICE_URL = 'https://masttest.stsci.edu'
    mast._service_api_connection.REQUEST_URL = 'https://masttest.stsci.edu/search/roman/api/v0.1/'
    mast._service_api_connection.MISSIONS_DOWNLOAD_URL = 'https://masttest.stsci.edu/search/'
    
    # update to use your own token from:
    # https://auth.masttest.stsci.edu/token?suggested_name=Astroquery&suggested_scope=mast:exclusive_access
    user_token = "your_masttest_token_here"
    my_session = mast.login(token=user_token)

    radius = 0.3 * u.deg
    roman_obs = mast.query_region(target, radius=radius)

## Create and display `mast-aladin`
Create a `MastAladin` instance and to display it in a sidecar split to the right.

Note the use of these options to enable skewer selection mode and the button that allows you to switch between that and edge selection.
```
    show_selection_mode_control=True,
    selection_mode='skewer',
```

In [ ]:
mast_aladin = MastAladin(
    fov=3,
    target=target,
    show_selection_mode_control=True,
    selection_mode='skewer',
    full_screen=True
)

# defaults to display in a sidecar split to the right
mast_aladin.show()

## Create and display `mast-table` 
Create a `MastTable` instance and to display it in a sidecar split to the bottom of the screen.

In [ ]:
mt = MastTable(roman_obs)
mt.show(loc='sidecar:split-bottom')

## Add `mast-table` as an overlay in `mast-aladin`
`MastAladin`'s `add_table()` method now accepts `MastTable`s. Supplying a `MastTable` causes `MastAladin` to automatically set up selection and filter synchronization between the `MastTable` instance and the corresponding table overlay in the `MastAladin` instance.

Explore selecting and filtering items in the `mast-table` and notice that those actions are reflected in Aladin. Items that are filtered out are then no longer selected.

Selecting footprints in `mast-aladin` selects those items in the `mast-table`. To leverage additive selections in `mast-aladin`, use Cmd-click (or Ctrl-click).

In [ ]:
roman_obs_layer = mast_aladin.add_table(
    mt,
    name='Roman Observations',
    color='#FF6600',
    ra_field="ra_ref",
    dec_field="dec_ref",
    mast_item_key="fileSetName"
)

## Using `.select()` in `mast-table`

We can use the API for `mast-table` to experiment with selection. `.select()` allows for selection based on:
* index values (int, list, slice)
* table masks (Boolean arrays)
* row values for a given column (value AND column name required)

In [ ]:
# index value list
mt.select([2,7,8])

In [ ]:
# index value slice
mt.select(slice(7,15))

In [ ]:
# Boolean table mask
mt.select(
    mt.table["dec_ref"] > 66
)

In [ ]:
# row values for given column
mt.select("WFI02", column = "detector")

In [ ]:
# remove all current selections
mt.select()

## About this notebook

**Last updated:** November 2026
***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 